In [7]:
import cv2 
from ultralytics import YOLO
import random

def process_video_with_tracking(model, input_video_path, show_video=True, save_video=False, output_video_path='output_video.mp4'):
  cap = cv2.VideoCapture(input_video_path)

  if not cap.isOpened():
    raise Exception('Error: Could not open video file.')
  
  fps = int(cap.get(cv2.CAP_PROP_FPS))
  frame_width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
  frame_height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

  if save_video:
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_video_path, fourcc, fps, (frame_width, frame_height))

    
  while True:
        ret, frame = cap.read()
        if not ret:
            break
        results = model.track(frame, iou=0.4, conf=0.5, persist=True, imgsz=608, verbose=False, tracker="bytetrack.yaml")

        if results[0].boxes.id != None: # this will ensure that id is not None -> exist tracks
            boxes = results[0].boxes.xyxy.cpu().numpy().astype(int)
            ids = results[0].boxes.id.cpu().numpy().astype(int)

            for box, id in zip(boxes, ids):
                # Generate a random color for each object based on its ID
                random.seed(int(id))
                color = (random.randint(0, 255), random.randint(0, 255), random.randint(0, 255))
                
                cv2.rectangle(frame, (box[0], box[1]), (box[2], box[3],), color, 2)
                cv2.putText(
                    frame,
                    f"Id {id}",
                    (box[0], box[1]),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.5,
                    (0, 255, 255),
                    2,
                )
        if save_video:
            out.write(frame)

        if show_video:
            frame = cv2.resize(frame, (0, 0), fx=0.75, fy=0.75)
            cv2.imshow("frame", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break
        
  if save_video:
        out.release()

    # Close all OpenCV windows
  cv2.destroyAllWindows()
  
model = YOLO('runs/detect/train/weights/best.pt')
model.fuse()
process_video_with_tracking(model, "video_cat.mp4", show_video=True, save_video=False, output_video_path="output_video.mp4")

    

Model summary (fused): 93 layers, 25,842,076 parameters, 0 gradients, 78.7 GFLOPs


AttributeError: 
            'IterableSimpleNamespace' object has no attribute 'fuse_score'. This may be caused by a modified or out of date ultralytics
            'default.yaml' file.
Please update your code with 'pip install -U ultralytics' and if necessary replace
            C:\Users\himyn\miniconda3\envs\WORK\Lib\site-packages\ultralytics\cfg\default.yaml with the latest version from
            https://github.com/ultralytics/ultralytics/blob/main/ultralytics/cfg/default.yaml
            